# Run `best_model1` to `best_model5` and export averaged solubility predictions

This notebook runs inference with `best_model1.pth` through `best_model5.pth`, writes each model's predicted solubility, computes the five-model average, and saves the result as an Excel file.

## Expected GitHub repository layout

Place this notebook in the same directory as the project files below:

```text
.
├── predict.ipynb
├── config.yaml
├── vocab.txt
├── Smiles_for_pre.xlsx
├── test_stage_norm.pt
├── best_model1.pth
├── best_model2.pth
├── best_model3.pth
├── best_model4.pth
├── best_model5.pth
├── dataset.py
├── tokenizer.py
├── models.py
└── Validation_and_test_supervised_1.py    # optional fallback for Normalizer
```

The output file will be:

```text
solubility_predictions.xlsx
```


## 1. Configuration

The paths below are relative to the notebook location by default. If your files are in another folder, change `PROJECT_DIR`.


In [18]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

CONFIG_PATH = PROJECT_DIR / "config.yaml"
VOCAB_PATH = PROJECT_DIR / "vocab.txt"
INPUT_EXCEL_PATH = PROJECT_DIR / "Smiles_for_pre.xlsx"
NORMALIZER_PATH = PROJECT_DIR / "test_stage_norm.pt"
MODEL_PATHS = [PROJECT_DIR / f"best_model{i}.pth" for i in range(1, 6)]
OUTPUT_EXCEL_PATH = PROJECT_DIR / "solubility_predictions.xlsx"

BATCH_SIZE = 1024
SHEET_NAME = 0
PRINT_COLLATE_TIME = False

print(f"Project directory: {PROJECT_DIR}")
print(f"Output Excel path: {OUTPUT_EXCEL_PATH}")

Project directory: D:\SSTrans\Article\code\Predict
Output Excel path: D:\SSTrans\Article\code\Predict\solubility_predictions.xlsx


## 2. Imports and reproducibility

In [19]:
import os
import random

import numpy as np

import yaml
import pandas as pd
from torch.utils.data import DataLoader
from dataset import dataset_generation
from torch.amp import autocast
import time
from tokenizer import SMILES_Atomwise_Tokenizer
from models import Whole_block
import torch

class Normalizer(object):
    """Normalize a Tensor and restore it later. """

    def __init__(self, tensor):
        """tensor is taken as a sample to calculate the mean and std"""
        tensor = tensor.float()
        self.mean = torch.mean(tensor)
        self.std = torch.std(tensor)

    def norm(self, tensor):
        return (tensor - self.mean) / self.std

    def denorm(self, normed_tensor):
        return normed_tensor * self.std + self.mean

    def state_dict(self):
        return {'mean': self.mean,
                'std': self.std}

    def load_state_dict(self, state_dict):
        self.mean = state_dict['mean']
        self.std = state_dict['std']


def setup_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def norm_temp(tensor: torch.Tensor) -> torch.Tensor:
    return 1 / tensor - 1 / 298


def denorm_temp(normed_tensor: torch.Tensor) -> torch.Tensor:
    return 1 / (normed_tensor + 1 / 298)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


## 3. Check required files

In [20]:
required_files = [
    CONFIG_PATH,
    VOCAB_PATH,
    INPUT_EXCEL_PATH,
    NORMALIZER_PATH,
    *MODEL_PATHS,
]

missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required files. Put these files in PROJECT_DIR or update the paths above:\n"
        + "\n".join(missing_files)
    )

print("All required files were found.")

All required files were found.


## 4. Load config, tokenizer, and normalizers

In [21]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

model_args = config["transformer"]
tokenizer = SMILES_Atomwise_Tokenizer(str(VOCAB_PATH))

normalizer_states = torch.load(NORMALIZER_PATH, map_location=device)
normalizers = {}
for feature, state in normalizer_states.items():
    normalizers[feature] = Normalizer(torch.tensor([0.0]))
    normalizers[feature].load_state_dict(state)

if "solubility" not in normalizers:
    raise KeyError("The loaded normalizers do not contain key 'solubility'.")

print("Config, tokenizer, and normalizers loaded successfully.")

Config, tokenizer, and normalizers loaded successfully.


C:\Users\admin\AppData\Local\Temp\ipykernel_30952\3815015723.py:23: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  self.std = torch.std(tensor)


## 5. Batch collation function

In [22]:
def collate_fn_labeled(input_batch):
    starter_time = time.time()

    Solute_SMILES = [item[0] for item in input_batch]
    Solvent_SMILES = [item[1] for item in input_batch]
    solubility = [item[2] for item in input_batch]
    temperature = [item[3] for item in input_batch]
    T_ref = [item[4] for item in input_batch]
    Solute_ref = [item[5] for item in input_batch]
    Solvent_ref = [item[6] for item in input_batch]

    solvation_free_energy = [item[7] for item in input_batch]
    solvation_enthalpy = [item[8] for item in input_batch]
    heat_capacity_cp = [item[9] for item in input_batch]
    heat_capacity_cs = [item[10] for item in input_batch]
    sublimation_enthalpy = [item[11] for item in input_batch]

    polarity_compatibility = [item[12] for item in input_batch]
    size_compatibility = [item[13] for item in input_batch]
    hbond_compatibility = [item[14] for item in input_batch]
    hydrophobicity_compatibility = [item[15] for item in input_batch]
    electrostatic_compatibility = [item[16] for item in input_batch]
    flexibility_compatibility = [item[17] for item in input_batch]
    aromaticity_compatibility = [item[18] for item in input_batch]
    charge_compatibility = [item[19] for item in input_batch]

    solubility_gradient = [item[20] for item in input_batch]
    result = {}

    if None not in Solute_SMILES:
        tokenized_Solute_SMILES = tokenizer(
            Solute_SMILES,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
        result["Solute_SMILES_ids"] = tokenized_Solute_SMILES["input_ids"]
        result["Solute_attention_mask"] = tokenized_Solute_SMILES["attention_mask"]
    else:
        result["Solute_SMILES_ids"] = None
        result["Solute_attention_mask"] = None

    if None not in Solvent_SMILES:
        tokenized_Solvent_SMILES = tokenizer(
            Solvent_SMILES,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
        result["Solvent_SMILES_ids"] = tokenized_Solvent_SMILES["input_ids"]
        result["Solvent_attention_mask"] = tokenized_Solvent_SMILES["attention_mask"]
    else:
        result["Solvent_SMILES_ids"] = None
        result["Solvent_attention_mask"] = None

    if None not in Solute_ref:
        tokenized_Solute_ref = tokenizer(
            Solute_ref,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
        result["Solute_ref"] = tokenized_Solute_ref["input_ids"]
        result["Solute_ref_mask"] = tokenized_Solute_ref["attention_mask"]
    else:
        result["Solute_ref"] = None
        result["Solute_ref_mask"] = None

    if None not in Solvent_ref:
        tokenized_Solvent_ref = tokenizer(
            Solvent_ref,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
        result["Solvent_ref"] = tokenized_Solvent_ref["input_ids"]
        result["Solvent_ref_mask"] = tokenized_Solvent_ref["attention_mask"]
    else:
        result["Solvent_ref"] = None
        result["Solvent_ref_mask"] = None

    def process_tensor(values, key_name):
        if None not in values:
            tensor = torch.tensor(values).unsqueeze(1)
            if key_name in normalizers:
                tensor = normalizers[key_name].norm(tensor)
            result[key_name] = tensor
        else:
            result[key_name] = None

    if None not in temperature:
        temperature_tensor = torch.tensor(temperature).unsqueeze(1)
        result["temperature"] = temperature_tensor
        result["temperature_norm"] = norm_temp(temperature_tensor)
    else:
        result["temperature"] = None
        result["temperature_norm"] = None

    if None not in T_ref:
        T_ref_tensor = torch.tensor(T_ref).unsqueeze(1)
        result["T_ref"] = T_ref_tensor
        result["T_ref_norm"] = norm_temp(T_ref_tensor)
    else:
        result["T_ref"] = None
        result["T_ref_norm"] = None

    process_tensor(solubility, "solubility")
    process_tensor(solvation_free_energy, "solvation_free_energy")
    process_tensor(solvation_enthalpy, "solvation_enthalpy")
    process_tensor(heat_capacity_cp, "heat_capacity_cp")
    process_tensor(heat_capacity_cs, "heat_capacity_cs")
    process_tensor(sublimation_enthalpy, "sublimation_enthalpy")
    process_tensor(polarity_compatibility, "polarity_compatibility")
    process_tensor(size_compatibility, "size_compatibility")
    process_tensor(hbond_compatibility, "hbond_compatibility")
    process_tensor(hydrophobicity_compatibility, "hydrophobicity_compatibility")
    process_tensor(electrostatic_compatibility, "electrostatic_compatibility")
    process_tensor(flexibility_compatibility, "flexibility_compatibility")
    process_tensor(aromaticity_compatibility, "aromaticity_compatibility")
    process_tensor(charge_compatibility, "charge_compatibility")
    process_tensor(solubility_gradient, "solubility_gradient")

    if PRINT_COLLATE_TIME:
        print(f"Total collate time: {time.time() - starter_time:.4f} seconds")

    return result

## 6. Load prediction data

In [23]:
pre_set = pd.read_excel(INPUT_EXCEL_PATH, sheet_name=SHEET_NAME)
dataset_pre = dataset_generation(pre_set)

pre_loader = DataLoader(
    dataset_pre,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn_labeled,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

print(f"Loaded prediction dataset with {len(dataset_pre)} samples.")
pre_set.head()

Loaded prediction dataset with 29 samples.


,SoluteSMILES,SolventSMILES,T_round
0,B#N,O,298
1,B12B3B4B1C234,O,298
2,BrCCBr,O,298
3,BrCCCBr,O,298
4,BrCCc1ccccc1,O,298


## 7. Run `best_model1` to `best_model5` and save the average

In [24]:
def move_required_batch_tensors_to_device(batch, device):
    required_keys = [
        "Solute_SMILES_ids",
        "Solvent_SMILES_ids",
        "Solute_attention_mask",
        "Solvent_attention_mask",
        "temperature",
    ]

    missing = [key for key in required_keys if batch.get(key) is None]
    if missing:
        raise ValueError(f"Batch is missing required tensors: {missing}")

    return [batch[key].to(device) for key in required_keys]


def predict_solubility_for_model(model, model_path: Path) -> torch.Tensor:
    print(f"Loading {model_path.name} ...")
    model_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(model_dict)
    model.eval()

    model_solubility_batches = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(pre_loader):
            model_inputs = move_required_batch_tensors_to_device(batch, device)
            with autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
                predictions = model(*model_inputs)

            solubility = predictions["solubility"]
            solubility = normalizers["solubility"].denorm(solubility)
            model_solubility_batches.append(solubility.detach().cpu())

    model_solubility = torch.cat(model_solubility_batches, dim=0).flatten()
    print(f"Finished {model_path.name}, prediction count: {len(model_solubility)}")
    return model_solubility


model = Whole_block(**model_args).to(device)
all_model_solubility = []

for model_path in MODEL_PATHS:
    model_solubility = predict_solubility_for_model(model, model_path)
    all_model_solubility.append(model_solubility)

# Shape: [num_samples, 5]
all_model_solubility = torch.stack(all_model_solubility, dim=1)
mean_solubility = all_model_solubility.mean(dim=1)

df_solubility = pd.DataFrame({
    "sample_id": range(len(mean_solubility)),
    "model1_predicted_solubility": all_model_solubility[:, 0].numpy(),
    "model2_predicted_solubility": all_model_solubility[:, 1].numpy(),
    "model3_predicted_solubility": all_model_solubility[:, 2].numpy(),
    "model4_predicted_solubility": all_model_solubility[:, 3].numpy(),
    "model5_predicted_solubility": all_model_solubility[:, 4].numpy(),
    "mean_predicted_solubility": mean_solubility.numpy(),
})

df_solubility.to_excel(OUTPUT_EXCEL_PATH, index=False)
print(f"Saved averaged predictions to: {OUTPUT_EXCEL_PATH}")

df_solubility.head()

Loading best_model1.pth ...
Finished best_model1.pth, prediction count: 29
Loading best_model2.pth ...
Finished best_model2.pth, prediction count: 29
Loading best_model3.pth ...
Finished best_model3.pth, prediction count: 29
Loading best_model4.pth ...
Finished best_model4.pth, prediction count: 29
Loading best_model5.pth ...
Finished best_model5.pth, prediction count: 29
Saved averaged predictions to: D:\SSTrans\Article\code\Predict\solubility_predictions.xlsx


,sample_id,model1_predicted_solubility,model2_predicted_solubility,model3_predicted_solubility,model4_predicted_solubility,model5_predicted_solubility,mean_predicted_solubility
0,0,-1.169922,-0.227539,0.326172,-0.986816,-0.203125,-0.452148
1,1,-2.605469,-2.027344,-1.687500,-2.251953,-1.067383,-1.927734
2,2,-1.731445,-1.316406,-1.996094,-1.992188,-1.826172,-1.772461
3,3,-2.083984,-2.277344,-1.828125,-2.232422,-2.107422,-2.105469
4,4,-3.208984,-3.246094,-3.234375,-3.074219,-3.117188,-3.175781


## 8. Check output file

In [25]:
if OUTPUT_EXCEL_PATH.exists():
    print(f"Output file exists: {OUTPUT_EXCEL_PATH}")
    print(f"File size: {OUTPUT_EXCEL_PATH.stat().st_size / 1024:.2f} KB")
else:
    raise FileNotFoundError(f"Output file was not created: {OUTPUT_EXCEL_PATH}")

Output file exists: D:\SSTrans\Article\code\Predict\solubility_predictions.xlsx
File size: 6.22 KB
